In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A01T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A04E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A08T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A03E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A09E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A06E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A06T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A02T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A05E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A03T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A07E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A09T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A08E.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A07T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A05T.gdf
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf/A02E.gdf
/kaggle/input/datasets/moonimint/bci-iv-

In [6]:
# Cell 1: Environment setup, downloading labels, and checking paths
!pip install braindecode -q

# Download true labels directly to working directory
!wget -q https://www.bbci.de/competition/iv/results/ds2a/true_labels.zip -O /kaggle/working/true_labels.zip
!unzip -q -o /kaggle/working/true_labels.zip -d /kaggle/working/true_labels

import os
import torch
import mne
import numpy as np
from scipy.io import loadmat

mne.set_log_level('WARNING')

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

print("\n--- Available Kaggle Inputs ---")
for dirname, _, filenames in os.walk('/kaggle/input'):
    if filenames:
        print(f"{dirname} -> {len(filenames)} files")

CUDA available: True
GPU name: Tesla T4

--- Available Kaggle Inputs ---
/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf -> 18 files


In [7]:
# Cell 2: Define Exact Epoching Functions for Raw GDF Files

dataset_path = '/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf'
true_labels_path = '/kaggle/working/true_labels'

def get_raw_epochs_T(gdf_path, verbose=False):
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, eog=['EOG-left', 'EOG-central', 'EOG-right'], verbose=verbose)
    events, event_id = mne.events_from_annotations(raw, verbose=verbose)
    
    target_event_id = {
        '769': event_id['769'], '770': event_id['770'], 
        '771': event_id['771'], '772': event_id['772']
    }
    
    epochs = mne.Epochs(raw, events, event_id=target_event_id, 
                        tmin=0, tmax=4.0, baseline=None, preload=True, verbose=verbose)
    epochs_eeg = epochs.copy().pick(picks='eeg') 
    
    drop_indices = []
    if '1023' in event_id:
        rejected_starts = events[events[:, 2] == event_id['1023']][:, 0]
        offset_samples = int(2.0 * raw.info['sfreq'])
        rejected_cue_samples = rejected_starts + offset_samples
        
        epoch_sample_times = epochs.events[:, 0]
        drop_mask = np.isin(epoch_sample_times, rejected_cue_samples)
        drop_indices = np.where(drop_mask)[0]
        epochs_eeg = epochs_eeg.drop(drop_indices, verbose=verbose)
        
    return epochs_eeg, target_event_id

def get_raw_epochs_E(gdf_path, mat_path, verbose=False):
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, eog=['EOG-left', 'EOG-central', 'EOG-right'], verbose=verbose)
    events, event_id = mne.events_from_annotations(raw, verbose=verbose)
    
    target_event_id = {'783': event_id['783']}
    
    epochs = mne.Epochs(raw, events, event_id=target_event_id, 
                        tmin=0, tmax=4.0, baseline=None, preload=True, verbose=verbose)
    epochs_eeg = epochs.copy().pick(picks='eeg')
    
    mat_data = loadmat(mat_path)
    labels = mat_data['classlabel'].flatten() 
    
    drop_indices = []
    if '1023' in event_id:
        rejected_starts = events[events[:, 2] == event_id['1023']][:, 0]
        offset_samples = int(2.0 * raw.info['sfreq'])
        rejected_cue_samples = rejected_starts + offset_samples
        
        epoch_sample_times = epochs.events[:, 0]
        drop_mask = np.isin(epoch_sample_times, rejected_cue_samples)
        drop_indices = np.where(drop_mask)[0]
        epochs_eeg = epochs_eeg.drop(drop_indices, verbose=verbose)
        labels = np.delete(labels, drop_indices) 
        
    assert len(epochs_eeg) == len(labels), "Mismatch between epochs and labels count!"
    return epochs_eeg, labels

print(f"Dataset path set to: {dataset_path}")
print("Epoching functions successfully defined!")

Dataset path set to: /kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf
Epoching functions successfully defined!


In [8]:
# Cell 3: Extract Raw Epochs and Labels for All Subjects

subjects = [f"{i:02d}" for i in range(1, 10)]

all_epochs = {}
all_labels = {}

print("Starting raw epoch extraction... This might take a minute.\n")

for subj in subjects:
    # --- T Session ---
    t_fpath = os.path.join(dataset_path, f"A{subj}T.gdf")
    if os.path.exists(t_fpath):
        ep, t_event_id = get_raw_epochs_T(t_fpath)
        # Map event codes (from GDF) to standard 1-4 class labels
        code_to_class = {t_event_id['769']: 1, t_event_id['770']: 2,
                         t_event_id['771']: 3, t_event_id['772']: 4}
        t_labels = np.array([code_to_class[e[2]] for e in ep.events])
        all_epochs[f"A{subj}T"] = ep
        all_labels[f"A{subj}T"] = t_labels
    else:
        print(f"Missing T session for A{subj}")
    
    # --- E Session ---
    e_fpath = os.path.join(dataset_path, f"A{subj}E.gdf")
    e_mat = os.path.join(true_labels_path, f"A{subj}E.mat")
    if os.path.exists(e_fpath) and os.path.exists(e_mat):
        ep, labels = get_raw_epochs_E(e_fpath, e_mat)
        all_epochs[f"A{subj}E"] = ep
        all_labels[f"A{subj}E"] = labels
    else:
        print(f"Missing E session or mat file for A{subj}")

print(f"\nDone! Successfully epoched: {len(all_epochs)}/18 files")

Starting raw epoch extraction... This might take a minute.



/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying runni


Done! Successfully epoched: 18/18 files


In [9]:
# Cell 4: Dataset Wrapper and Training Function
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import Dataset, DataLoader

class EEGDataset(Dataset):
    def __init__(self, X, y):
        # Convert to float32 for data, long/int64 for labels
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) 

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def train_model(model, train_loader, val_loader, n_epochs=200, lr=0.001, patience=25, weight_decay=0.01, verbose=False):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        # Training Phase
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total

        # Validation Phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if verbose and (epoch % 10 == 0 or epoch == n_epochs - 1):
            print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Early Stopping Check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_state)
    return model, history

print("PyTorch Dataset and Training loop ready!")

PyTorch Dataset and Training loop ready!


In [10]:
# Cell 5: Run Final Evaluation Pipeline (Train on T, Test on E)
def run_raw_te_pipeline(subject_id, n_epochs=200, lr=0.001, patience=25, weight_decay=0.01, drop_prob=0.5, batch_size=32, val_frac=0.2, seed=42):
    X_train_full = all_epochs[f"A{subject_id}T"].get_data()
    y_train_full = all_labels[f"A{subject_id}T"] - 1
    X_test = all_epochs[f"A{subject_id}E"].get_data()
    y_test = all_labels[f"A{subject_id}E"] - 1

    # Trial-level train/val split
    n_trials = len(X_train_full)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n_trials)
    n_val = int(val_frac * n_trials)
    val_idx, train_idx = indices[:n_val], indices[n_val:]

    X_tr, y_tr = X_train_full[train_idx], y_train_full[train_idx]
    X_val, y_val = X_train_full[val_idx], y_train_full[val_idx]

    # Normalize strictly on Training split
    mean = X_tr.mean(axis=(0, 2), keepdims=True)
    std = X_tr.std(axis=(0, 2), keepdims=True)
    X_tr_norm = (X_tr - mean) / (std + 1e-8)
    X_val_norm = (X_val - mean) / (std + 1e-8)
    X_test_norm = (X_test - mean) / (std + 1e-8)

    train_loader = DataLoader(EEGDataset(X_tr_norm, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val_norm, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(EEGDataset(X_test_norm, y_test), batch_size=batch_size, shuffle=False)

    # Initialize ShallowConvNet
    model = ShallowFBCSPNet(
        n_chans=22, 
        n_outputs=4, 
        n_times=X_train_full.shape[2],
        drop_prob=drop_prob,
        final_conv_length='auto'
    )

    trained, hist = train_model(model, train_loader, val_loader, n_epochs=n_epochs,
                                  lr=lr, patience=patience, weight_decay=weight_decay, verbose=False)

    # Evaluate on Unseen E-Session Test Set
    trained.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device)
            out = trained(Xb)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())

    test_acc = accuracy_score(trues, preds)
    test_kappa = cohen_kappa_score(trues, preds)
    best_val_acc = max(hist['val_acc'])

    print(f"Subject A{subject_id}: best_val_acc={best_val_acc:.4f} | test_acc={test_acc:.4f} | test_kappa={test_kappa:.4f}")

    return {
        'subject': f"A{subject_id}",
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_kappa': test_kappa
    }

print("Running Raw Data T->E Pipeline across all subjects...\n")
raw_te_results = []
for subj in subjects:
    res = run_raw_te_pipeline(subj)
    raw_te_results.append(res)

raw_te_df = pd.DataFrame(raw_te_results)
print("\n=== Final Test Set Summary (T->E) for ShallowConvNet [RAW DATA] ===")
print(raw_te_df)
print(f"\nMean test accuracy: {raw_te_df['test_acc'].mean():.4f}")
print(f"Mean test kappa: {raw_te_df['test_kappa'].mean():.4f}")

Running Raw Data T->E Pipeline across all subjects...



NameError: name 'device' is not defined

In [11]:
# 1. INSTALLS & IMPORTS
!pip install braindecode -q
!wget -q https://www.bbci.de/competition/iv/results/ds2a/true_labels.zip -O /kaggle/working/true_labels.zip
!unzip -q -o /kaggle/working/true_labels.zip -d /kaggle/working/true_labels

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy as copy_module
import mne
from scipy.io import loadmat
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, cohen_kappa_score
from braindecode.models import ShallowFBCSPNet

mne.set_log_level('WARNING')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# 2. PATHS & EXTRACTION FUNCTIONS
dataset_path = '/kaggle/input/datasets/moonimint/bci-iv-2a-raw-gdf'
true_labels_path = '/kaggle/working/true_labels'

def get_raw_epochs_T(gdf_path, verbose=False):
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, eog=['EOG-left', 'EOG-central', 'EOG-right'], verbose=verbose)
    events, event_id = mne.events_from_annotations(raw, verbose=verbose)
    target_event_id = {'769': event_id['769'], '770': event_id['770'], '771': event_id['771'], '772': event_id['772']}
    epochs = mne.Epochs(raw, events, event_id=target_event_id, tmin=0, tmax=4.0, baseline=None, preload=True, verbose=verbose)
    epochs_eeg = epochs.copy().pick(picks='eeg')
    
    drop_indices = []
    if '1023' in event_id:
        rejected_starts = events[events[:, 2] == event_id['1023']][:, 0]
        offset_samples = int(2.0 * raw.info['sfreq'])
        rejected_cue_samples = rejected_starts + offset_samples
        drop_mask = np.isin(epochs.events[:, 0], rejected_cue_samples)
        drop_indices = np.where(drop_mask)[0]
        epochs_eeg = epochs_eeg.drop(drop_indices, verbose=verbose)
    return epochs_eeg, target_event_id

def get_raw_epochs_E(gdf_path, mat_path, verbose=False):
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, eog=['EOG-left', 'EOG-central', 'EOG-right'], verbose=verbose)
    events, event_id = mne.events_from_annotations(raw, verbose=verbose)
    target_event_id = {'783': event_id['783']}
    epochs = mne.Epochs(raw, events, event_id=target_event_id, tmin=0, tmax=4.0, baseline=None, preload=True, verbose=verbose)
    epochs_eeg = epochs.copy().pick(picks='eeg')
    
    mat_data = loadmat(mat_path)
    labels = mat_data['classlabel'].flatten()
    
    drop_indices = []
    if '1023' in event_id:
        rejected_starts = events[events[:, 2] == event_id['1023']][:, 0]
        offset_samples = int(2.0 * raw.info['sfreq'])
        rejected_cue_samples = rejected_starts + offset_samples
        drop_mask = np.isin(epochs.events[:, 0], rejected_cue_samples)
        drop_indices = np.where(drop_mask)[0]
        epochs_eeg = epochs_eeg.drop(drop_indices, verbose=verbose)
        labels = np.delete(labels, drop_indices)
    
    assert len(epochs_eeg) == len(labels)
    return epochs_eeg, labels

# 3. BUILD ALL_EPOCHS DICTIONARY
subjects = [f"{i:02d}" for i in range(1, 10)]
all_epochs, all_labels = {}, {}
print("Extracting raw epochs for all 9 subjects...")
for subj in subjects:
    t_fpath, e_fpath = os.path.join(dataset_path, f"A{subj}T.gdf"), os.path.join(dataset_path, f"A{subj}E.gdf")
    e_mat = os.path.join(true_labels_path, f"A{subj}E.mat")
    
    ep_t, t_event_id = get_raw_epochs_T(t_fpath)
    code_to_class = {t_event_id['769']: 1, t_event_id['770']: 2, t_event_id['771']: 3, t_event_id['772']: 4}
    all_epochs[f"A{subj}T"] = ep_t
    all_labels[f"A{subj}T"] = np.array([code_to_class[e[2]] for e in ep_t.events])
    
    ep_e, labels_e = get_raw_epochs_E(e_fpath, e_mat)
    all_epochs[f"A{subj}E"], all_labels[f"A{subj}E"] = ep_e, labels_e

print(f"Extraction complete! Restored {len(all_epochs)} files into memory.")

Device: cuda
Extracting raw epochs for all 9 subjects...


/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/usr/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying runni

Extraction complete! Restored 18 files into memory.


In [12]:
# 1. PYTORCH UTILS
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def train_model(model, train_loader, val_loader, n_epochs=200, lr=0.001, patience=25, weight_decay=0.01, verbose=False):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_loss, best_model_state, patience_counter = float('inf'), None, 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(n_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss, train_acc = train_loss / train_total, train_correct / train_total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss, val_acc = val_loss / val_total, val_correct / val_total
        history['train_loss'].append(train_loss); history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss, best_model_state, patience_counter = val_loss, copy_module.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    model.load_state_dict(best_model_state)
    return model, history

def crop_array(X, y, crop_len=500, stride=50):
    cropped_X, cropped_y, trial_ids = [], [], []
    for i in range(len(X)):
        start = 0
        while start + crop_len <= X.shape[2]:
            cropped_X.append(X[i][:, start:start+crop_len])
            cropped_y.append(y[i])
            trial_ids.append(i)
            start += stride
    return np.array(cropped_X), np.array(cropped_y), np.array(trial_ids)

# 2. CROSS-SUBJECT PRETRAINING
crop_len, stride = 500, 50
all_pretrain_X, all_pretrain_y = [], []

print("Cropping and combining T-sessions for Pretraining...")
for subj in subjects:
    X, y = all_epochs[f"A{subj}T"].get_data(), all_labels[f"A{subj}T"] - 1
    X_c, y_c, _ = crop_array(X, y, crop_len, stride)
    all_pretrain_X.append(X_c); all_pretrain_y.append(y_c)

X_pretrain, y_pretrain = np.concatenate(all_pretrain_X, axis=0), np.concatenate(all_pretrain_y, axis=0)

n_total = len(X_pretrain)
idx = np.random.RandomState(42).permutation(n_total)
n_val = int(0.1 * n_total)
val_idx, train_idx = idx[:n_val], idx[n_val:]
X_pre_tr, y_pre_tr = X_pretrain[train_idx], y_pretrain[train_idx]
X_pre_val, y_pre_val = X_pretrain[val_idx], y_pretrain[val_idx]

pretrain_mean, pretrain_std = X_pre_tr.mean(axis=(0, 2), keepdims=True), X_pre_tr.std(axis=(0, 2), keepdims=True)
X_pre_tr_norm, X_pre_val_norm = (X_pre_tr - pretrain_mean) / (pretrain_std + 1e-8), (X_pre_val - pretrain_mean) / (pretrain_std + 1e-8)

pretrain_loader = DataLoader(EEGDataset(X_pre_tr_norm, y_pre_tr), batch_size=128, shuffle=True)
pretrain_val_loader = DataLoader(EEGDataset(X_pre_val_norm, y_pre_val), batch_size=128, shuffle=False)

pretrained_model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=0.5)
print("Running Cross-Subject Pretraining (up to 100 epochs)...")
pretrained_model, pretrain_hist = train_model(pretrained_model, pretrain_loader, pretrain_val_loader, n_epochs=100, lr=0.001, patience=15)
pretrained_state = pretrained_model.state_dict()
print(f"Pretraining Done! Best Val Acc: {max(pretrain_hist['val_acc']):.4f}")

Cropping and combining T-sessions for Pretraining...
Running Cross-Subject Pretraining (up to 100 epochs)...
Pretraining Done! Best Val Acc: 0.7055


In [13]:
import pandas as pd

def finetune_subject(subject_id, pretrained_state_dict, crop_len=500, stride=50, n_epochs=50, lr=0.0001, patience=15, drop_prob=0.5, batch_size=32, val_frac=0.2, seed=42):
    X_train_full, y_train_full = all_epochs[f"A{subject_id}T"].get_data(), all_labels[f"A{subject_id}T"] - 1
    X_test_full, y_test_full = all_epochs[f"A{subject_id}E"].get_data(), all_labels[f"A{subject_id}E"] - 1

    n_trials = len(X_train_full)
    indices = np.random.RandomState(seed).permutation(n_trials)
    n_val = int(val_frac * n_trials)
    val_idx, train_idx = indices[:n_val], indices[n_val:]

    X_tr_trials, y_tr_trials = X_train_full[train_idx], y_train_full[train_idx]
    X_val_trials, y_val_trials = X_train_full[val_idx], y_train_full[val_idx]

    X_tr_c, y_tr_c, _ = crop_array(X_tr_trials, y_tr_trials, crop_len, stride)
    X_val_c, y_val_c, _ = crop_array(X_val_trials, y_val_trials, crop_len, stride)

    X_tr_norm, X_val_norm = (X_tr_c - pretrain_mean) / (pretrain_std + 1e-8), (X_val_c - pretrain_mean) / (pretrain_std + 1e-8)

    train_loader = DataLoader(EEGDataset(X_tr_norm, y_tr_c), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val_norm, y_val_c), batch_size=batch_size, shuffle=False)

    model = ShallowFBCSPNet(n_chans=22, n_outputs=4, n_times=crop_len, drop_prob=drop_prob)
    model.load_state_dict(copy_module.deepcopy(pretrained_state_dict))

    finetuned, hist = train_model(model, train_loader, val_loader, n_epochs=n_epochs, lr=lr, patience=patience)

    X_test_c, y_test_c, test_trial_ids = crop_array(X_test_full, y_test_full, crop_len, stride)
    X_test_norm = (X_test_c - pretrain_mean) / (pretrain_std + 1e-8)
    test_loader = DataLoader(EEGDataset(X_test_norm, y_test_c), batch_size=batch_size, shuffle=False)

    finetuned.eval()
    crop_preds = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            out = finetuned(Xb.to(device))
            crop_preds.extend(out.argmax(1).cpu().numpy())
    crop_preds = np.array(crop_preds)

    final_preds = [np.bincount(crop_preds[test_trial_ids == trial_i]).argmax() for trial_i in range(len(X_test_full))]
    
    test_acc = accuracy_score(y_test_full, final_preds)
    test_kappa = cohen_kappa_score(y_test_full, final_preds)
    best_val_acc = max(hist['val_acc'])
    print(f"Subject A{subject_id}: best_val_acc={best_val_acc:.4f} | test_acc={test_acc:.4f} | test_kappa={test_kappa:.4f}")
    return {'subject': f"A{subject_id}", 'best_val_acc': best_val_acc, 'test_acc': test_acc, 'test_kappa': test_kappa}

print("Running Fine-Tuning and Eval for all subjects...\n")
results_df = pd.DataFrame([finetune_subject(subj, pretrained_state) for subj in subjects])

print("\n=== Final Fine-Tuned Test Set Summary [RAW DATA] ===")
print(results_df)
print(f"\nMean test accuracy: {results_df['test_acc'].mean():.4f}")
print(f"Mean test kappa: {results_df['test_kappa'].mean():.4f}")

Running Fine-Tuning and Eval for all subjects...

Subject A01: best_val_acc=0.8603 | test_acc=0.8541 | test_kappa=0.8054
Subject A02: best_val_acc=0.7037 | test_acc=0.4311 | test_kappa=0.2422
Subject A03: best_val_acc=0.8586 | test_acc=0.8462 | test_kappa=0.7948
Subject A04: best_val_acc=0.6923 | test_acc=0.5658 | test_kappa=0.4244
Subject A05: best_val_acc=0.7815 | test_acc=0.2645 | test_kappa=0.0369
Subject A06: best_val_acc=0.6512 | test_acc=0.6047 | test_kappa=0.4723
Subject A07: best_val_acc=0.9024 | test_acc=0.8809 | test_kappa=0.8412
Subject A08: best_val_acc=0.8671 | test_acc=0.8118 | test_kappa=0.7493
Subject A09: best_val_acc=0.8104 | test_acc=0.7614 | test_kappa=0.6817

=== Final Fine-Tuned Test Set Summary [RAW DATA] ===
  subject  best_val_acc  test_acc  test_kappa
0     A01      0.860269  0.854093    0.805379
1     A02      0.703704  0.431095    0.242183
2     A03      0.858586  0.846154    0.794770
3     A04      0.692308  0.565789    0.424419
4     A05      0.781469  0.